# RSnowflake -- Feature Demo

An interactive walkthrough of **RSnowflake** (multi-backend DBI connector
for Snowflake) running inside a Snowflake Workspace Notebook.

**Architecture:** REST API v2 (baseline) + optional ADBC acceleration
(Arrow-native reads, PUT+COPY INTO writes) when `adbcsnowflake` is installed.

**Sections:**
1. Setup (R environment + PAT auth)
2. Connect
3. Simple queries & type mapping
4. Grid viewer for R data.frames
5. Table operations (write, read, append)
6. Identifier case handling
7. Parameterized queries
8. Transactions
9. dbplyr / dplyr integration
10. Lazy dbplyr grid display
11. Snowpark to dplyr (Python to R)
12. Arrow fast path
13. ADBC backend acceleration (reads and writes)
14. Database object browsing (dbListObjects)
15. Cleanup

## 1. Setup

Install the R environment (skip if already done in this session),
register the `%%R` magic, create a PAT, and push session context to env vars.

In [ ]:
# Install R environment with ADBC and register %%R magic
from sfnb_setup import install_r
install_r(r_adbc=True)

In [ ]:
%%R
# Verify RSnowflake dependencies (installed by setup script)
pkgs <- c("DBI", "httr2", "jsonlite", "rlang", "cli",
          "dbplyr", "dplyr", "nanoarrow")
missing <- pkgs[!vapply(pkgs, requireNamespace, logical(1), quietly = TRUE)]
if (length(missing)) {
  cat("Installing missing core deps:", paste(missing, collapse = ", "), "\n")
  install.packages(missing, repos = "https://cloud.r-project.org")
}
cat("Core dependencies OK.\n")

# ADBC status (installed by setup script --adbc flag)
cat("adbcdrivermanager:", requireNamespace("adbcdrivermanager", quietly = TRUE), "\n")
cat("adbcsnowflake:    ", requireNamespace("adbcsnowflake", quietly = TRUE), "\n")
if (!requireNamespace("adbcsnowflake", quietly = TRUE)) {
  cat("NOTE: ADBC not installed. Re-run setup with --adbc for Arrow-native performance.\n")
}

In [ ]:
%%R
if ("RSnowflake" %in% loadedNamespaces()) {
  try(detach("package:RSnowflake", unload = TRUE), silent = TRUE)
  unloadNamespace("RSnowflake")
}

if (nzchar(Sys.getenv("RSNOWFLAKE_PATH"))) {
  install.packages(Sys.getenv("RSNOWFLAKE_PATH"), repos = NULL, type = "source")
} else {
  install.packages("pak", type = "source", quiet = TRUE)
  pak::pak("Snowflake-Labs/RSnowflake", ask = FALSE, upgrade = FALSE)
}

library(RSnowflake)
cat("RSnowflake", as.character(packageVersion("RSnowflake")), "loaded\n")

In [ ]:
import os
from snowflake.snowpark.context import get_active_session
from r_helpers import PATManager

session = get_active_session()
pat_mgr = PATManager(session)

if os.environ.get("SNOWFLAKE_PAT"):
    print("PAT already set in environment -- skipping creation.")
    print(f"SNOWFLAKE_PAT set: True (len={len(os.environ['SNOWFLAKE_PAT'])})")
else:
    result = pat_mgr.create_pat(days_to_expiry=1, force_recreate=True)
    if result['success']:
        print(f"PAT created for {result['user']} (role: {result['role_restriction']})")
        print(f"Expires: {result['expires_at']}")
    else:
        print(f"PAT creation failed: {result['error']}")

In [ ]:
import os

os.environ["SNOWFLAKE_ACCOUNT"]   = session.get_current_account().replace('"', '')
os.environ["SNOWFLAKE_USER"]      = session.sql("SELECT CURRENT_USER()").collect()[0][0]
os.environ["SNOWFLAKE_DATABASE"]  = (session.get_current_database() or "").replace('"', '')
os.environ["SNOWFLAKE_SCHEMA"]    = (session.get_current_schema() or "").replace('"', '')
os.environ["SNOWFLAKE_WAREHOUSE"] = (session.get_current_warehouse() or "").replace('"', '')
os.environ["SNOWFLAKE_ROLE"]      = (session.get_current_role() or "").replace('"', '')

for k in ["SNOWFLAKE_ACCOUNT", "SNOWFLAKE_DATABASE", "SNOWFLAKE_WAREHOUSE", "SNOWFLAKE_ROLE"]:
    print(f"{k}: {os.environ[k]}")

## 2. Connect

In [ ]:
%%R
if (!nzchar(Sys.getenv("TZ", ""))) Sys.setenv(TZ = "UTC")
options(width = 200)

library(DBI)
library(RSnowflake)

con <- dbConnect(Snowflake())
con
dbGetInfo(con)

## 3. Simple Queries & Type Mapping

In [ ]:
%%R
dbGetQuery(con, "SELECT CURRENT_VERSION() AS version")

In [ ]:
%%R
dbGetQuery(con, "
  SELECT
    42            AS int_val,
    3.14::DOUBLE  AS dbl_val,
    'hello'       AS str_val,
    TRUE          AS bool_val,
    CURRENT_DATE()          AS date_val,
    CURRENT_TIMESTAMP()     AS ts_val
")

## 4. Grid Viewer for R Data.frames

When a `%%R` cell's last visible expression is a `data.frame` (or tibble),
it is **automatically** converted to a pandas DataFrame and returned to
Workspace, which renders the interactive grid viewer -- just like Python cells.

Lazy `dbplyr` tables are also auto-detected: their SQL is executed via
Snowpark directly, avoiding R-to-Python data copy.

The query in the previous cell already demonstrates this: the grid appears
automatically because `dbGetQuery()` returns a data.frame.

**Flags:**
- `%%R --text` -- opt out; force plain text output
- `%%R --df varname` -- explicitly name which R variable to display in the grid
- `%%R -i pyvar` -- pass Python variables into R (Snowpark DataFrames become lazy dbplyr tables automatically)

In [ ]:
%%R --df result
result <- dbGetQuery(con, "
  SELECT 42 AS INT_VAL, 3.14::DOUBLE AS DBL_VAL, 'hello' AS STR_VAL
")
cat("Query returned", nrow(result), "row(s)\n")
cat("Grid viewer shows the 'result' variable below (via --df flag)")


In [ ]:
%%R --text
dbGetQuery(con, "SELECT 1 AS ID, 'text mode' AS NOTE")

## 5. Table Operations

Write a demo data.frame, read it back, and append more rows.
Column names are uppercased by default (standard Snowflake behaviour).

In [ ]:
%%R
demo <- data.frame(
  id     = 1:10,
  city   = c("London", "Paris", "Tokyo", "Sydney", "NYC",
             "Berlin", "Toronto", "Mumbai", "Seoul", "Dubai"),
  temp_c = c(12.5, 15.2, 22.3, 25.1, 18.7,
             10.3, 8.9, 33.2, 19.8, 38.5),
  rainy  = c(TRUE, TRUE, FALSE, FALSE, TRUE,
             TRUE, TRUE, FALSE, FALSE, FALSE),
  stringsAsFactors = FALSE
)

dbWriteTable(con, "DEMO_CITIES", demo, overwrite = TRUE)
cat("Table created.\n")

# Column names are uppercased by default
dbListFields(con, "DEMO_CITIES")

In [ ]:
%%R
dbReadTable(con, "DEMO_CITIES")

In [ ]:
%%R
extra <- data.frame(
  id = 11:12,
  city = c("Rome", "Cairo"),
  temp_c = c(20.1, 35.0),
  rainy = c(FALSE, FALSE)
)
dbAppendTable(con, "DEMO_CITIES", extra)

dbGetQuery(con, "SELECT COUNT(*) AS n FROM DEMO_CITIES")

## 6. Identifier Case Handling

By default, RSnowflake uppercases table and column names to match
Snowflake convention. In raw SQL you can reference them unquoted
(Snowflake auto-uppercases) or with uppercase quoted identifiers.

In [ ]:
%%R
# Columns are uppercase -- unquoted names work, or use uppercase quoted identifiers
dbGetQuery(con, 'SELECT CITY, TEMP_C FROM DEMO_CITIES WHERE TEMP_C > 25')

In [ ]:
%%R
# dbQuoteIdentifier wraps names in double-quotes
dbQuoteIdentifier(con, "myColumn")

# dbUnquoteIdentifier parses back
dbUnquoteIdentifier(con, SQL('"mydb"."myschema"."mytable"'))

## 7. Parameterized Queries

Use `?` placeholders with `params` or `dbBind`.

In [ ]:
%%R
dbGetQuery(
  con,
  'SELECT * FROM DEMO_CITIES WHERE TEMP_C > ?',
  params = list(30)
)

In [ ]:
%%R
res <- dbSendQuery(con, 'SELECT * FROM DEMO_CITIES WHERE CITY = ?')
dbBind(res, list("Tokyo"))
dbFetch(res)
dbClearResult(res)

## 8. Transactions (not yet supported)

The Snowflake SQL API v2 is stateless per-request, so session-based
transactions (`dbBegin`/`dbCommit`/`dbRollback`) are not yet supported.
This section demonstrates that RSnowflake reports a clear error.

In [ ]:
%%R
# Manual transaction -- expected to fail (SQL API v2 is stateless)
tryCatch(
  dbBegin(con),
  error = function(e) cat("Expected:", conditionMessage(e), "\n")
)

In [ ]:
%%R
# dbWithTransaction -- also expected to fail
tryCatch(
  dbWithTransaction(con, {
    dbExecute(con, "SELECT 1")
  }),
  error = function(e) cat("Expected:", conditionMessage(e), "\n")
)

## 9. dbplyr / dplyr Integration

If `dbplyr` and `dplyr` are available, queries can be composed with
familiar tidyverse verbs and translated to Snowflake SQL lazily.

In [ ]:
%%R
if (requireNamespace("dbplyr", quietly = TRUE) &&
    requireNamespace("dplyr", quietly = TRUE)) {

  library(dplyr)

  cities_tbl <- tbl(con, "DEMO_CITIES")

  # Lazy query -- translated to Snowflake SQL, not executed yet
  hot_cities <- cities_tbl |>
    filter(TEMP_C > 20) |>
    select(CITY, TEMP_C) |>
    arrange(desc(TEMP_C))

  cat("== Generated SQL ==\n")
  show_query(hot_cities)

  cat("\n== Results ==\n")
  print(hot_cities |> collect())

  cat("\n== Aggregation ==\n")
  print(
    cities_tbl |>
      summarise(
        avg_temp = mean(TEMP_C, na.rm = TRUE),
        n_rainy  = sum(as.integer(RAINY), na.rm = TRUE),
        n_cities = n()
      ) |>
      collect()
  )

} else {
  cat("Skipped: install dbplyr and dplyr for this section.\n")
}

## 10. Lazy dbplyr Grid Display

When a `%%R` cell's last visible expression is a **lazy dbplyr table**
(`tbl_lazy`), the magic automatically extracts the generated SQL and
executes it directly via the Snowpark session -- no R-to-Python data
copy needed. The result is displayed in the Workspace grid viewer.

In [ ]:
%%R
library(dplyr)

# Build a lazy query -- nothing is sent to Snowflake yet
hot_cities <- tbl(con, "DEMO_CITIES") |>
  filter(TEMP_C > 20) |>
  select(CITY, TEMP_C) |>
  arrange(desc(TEMP_C))

cat("Generated SQL:\n")
show_query(hot_cities)

# Return the lazy table as the last expression -- auto-displayed as grid
cat("\nGrid viewer (executed via Snowpark, zero R→Python copy):\n")
hot_cities

## 11. Snowpark to dplyr (Python to R)

Use `%%R -i pyvar` to push a **Snowpark DataFrame** into R as a lazy
`dbplyr` table. The magic auto-detects that the variable is a Snowpark
DataFrame, extracts its SQL, and wraps it as `tbl(con, sql(...))` --
no data is materialised until you `collect()`. Regular Python objects
on the same `-i` are passed through rpy2's native converter as usual.

Database/schema context is checked automatically: if the Snowpark and R
sessions have different contexts, the R connection is aligned before
creating the lazy table.

`--snow pyvar[=r_name]` is also available as an explicit alternative.

In [ ]:
# Build a Snowpark DataFrame in Python
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import col

session = get_active_session()
warm_cities = session.table("DEMO_CITIES").filter(col("TEMP_C") > 15)

print("Snowpark DataFrame SQL:")
print(warm_cities.queries['queries'][-1])

In [ ]:
%%R -i warm_cities
# warm_cities is auto-detected as a Snowpark DF and injected as a
# lazy dbplyr tbl backed by the Snowpark query's SQL.
# Continue refining with dplyr verbs -- still lazy, no data pulled yet.
library(dplyr)

result <- warm_cities |>
  select(CITY, TEMP_C) |>
  arrange(TEMP_C)

cat("Generated SQL (Snowpark filter + dplyr select/arrange):\n")
show_query(result)

# Return lazy table as last expression -- auto-displayed as grid
result

## 12. Arrow Fast Path (optional)

If `nanoarrow` is installed, RSnowflake can stream results in Arrow format
for lower overhead on large result sets.

In [ ]:
%%R
if (requireNamespace("nanoarrow", quietly = TRUE)) {
  stream <- dbGetQueryArrow(con, "SELECT * FROM DEMO_CITIES")
  arrow_df <- as.data.frame(stream)
  cat("Arrow result:", nrow(arrow_df), "rows,", ncol(arrow_df), "columns\n")
  arrow_df
} else {
  cat("Skipped: install nanoarrow for Arrow fast path.\n")
}

## 13. ADBC Backend Acceleration

When `adbcsnowflake` is installed, RSnowflake transparently routes queries
through the Snowflake ADBC Go driver for significantly better performance:

- **Reads:** Arrow-native transport (no JSON serialisation overhead)
- **Writes:** PUT + COPY INTO (bulk ingest, not row-by-row INSERT)

The ADBC backend connects to the **public endpoint** using PAT auth
(external-client pattern). In `"auto"` mode (the default), RSnowflake
uses ADBC when available and falls back to REST API v2 otherwise.

| Option | Values | Description |
| --- | --- | --- |
| `RSnowflake.backend` | `"auto"`, `"adbc"`, `"rest"` | Which backend to use for reads |
| `RSnowflake.upload_method` | `"auto"`, `"adbc"`, `"literal"` | Which backend to use for writes |

In [ ]:
%%R
has_adbc <- requireNamespace("adbcsnowflake", quietly = TRUE) &&
            requireNamespace("adbcdrivermanager", quietly = TRUE)
cat("ADBC packages installed:", has_adbc, "\n")

if (has_adbc) {
  # Warm up the ADBC backend (first call includes Go driver init + TLS + PAT auth).
  # This isolates init cost from the actual read benchmark.
  options(RSnowflake.backend = "adbc")
  t0 <- proc.time()
  warmup <- dbGetQuery(con, "SELECT 1 AS x")
  init_time <- (proc.time() - t0)[["elapsed"]]
  cat(sprintf("ADBC init:  %.2fs (one-time cost, includes Go driver + PAT handshake)\n", init_time))

  # --- ADBC read (Arrow-native, steady-state) ---
  t0 <- proc.time()
  adbc_df <- dbGetQuery(con, "
    SELECT SEQ4() AS id, RANDSTR(20, RANDOM()) AS txt,
           UNIFORM(0::FLOAT, 100::FLOAT, RANDOM()) AS score
    FROM TABLE(GENERATOR(ROWCOUNT => 50000))
  ")
  adbc_time <- (proc.time() - t0)[["elapsed"]]

  # --- REST read (JSON) ---
  options(RSnowflake.backend = "rest")
  t0 <- proc.time()
  rest_df <- dbGetQuery(con, "
    SELECT SEQ4() AS id, RANDSTR(20, RANDOM()) AS txt,
           UNIFORM(0::FLOAT, 100::FLOAT, RANDOM()) AS score
    FROM TABLE(GENERATOR(ROWCOUNT => 50000))
  ")
  rest_time <- (proc.time() - t0)[["elapsed"]]

  options(RSnowflake.backend = "auto")

  cat(sprintf("\nADBC read:  50K rows in %.2fs (steady-state)\n", adbc_time))
  cat(sprintf("REST read:  50K rows in %.2fs\n", rest_time))
  cat(sprintf("Speedup:    %.1fx\n", rest_time / adbc_time))
} else {
  cat("Skipped: install adbcsnowflake for ADBC acceleration.\n")
}

In [ ]:
%%R
if (has_adbc) {
  in_ws <- nzchar(Sys.getenv("SNOWFLAKE_HOST", ""))
  has_snowpark <- tryCatch({
    requireNamespace("reticulate", quietly = TRUE) &&
      !is.null(reticulate::import("snowflake.snowpark", delay_load = FALSE))
  }, error = function(e) FALSE)

  write_5k <- data.frame(
    id = 1:5000, txt = paste0("row_", 1:5000), val = runif(5000),
    stringsAsFactors = FALSE
  )
  write_50k <- data.frame(
    id = 1:50000, txt = paste0("row_", 1:50000), val = runif(50000),
    stringsAsFactors = FALSE
  )

  results <- data.frame(
    method = character(), rows = character(), time_s = numeric(),
    stringsAsFactors = FALSE
  )

  for (n_label in c("5K", "50K")) {
    df <- if (n_label == "5K") write_5k else write_50k
    tbl <- paste0("WRITE_BENCH_", n_label)

    # Literal INSERT
    options(RSnowflake.upload_method = "literal")
    t0 <- proc.time()
    dbWriteTable(con, tbl, df, overwrite = TRUE)
    lit_t <- (proc.time() - t0)[["elapsed"]]
    results <- rbind(results, data.frame(method = "literal", rows = n_label, time_s = lit_t))
    dbExecute(con, paste("DROP TABLE IF EXISTS", tbl))

    # ADBC
    options(RSnowflake.upload_method = "adbc")
    t0 <- proc.time()
    dbWriteTable(con, tbl, df, overwrite = TRUE)
    adbc_t <- (proc.time() - t0)[["elapsed"]]
    cnt <- dbGetQuery(con, paste("SELECT COUNT(*) AS N FROM", tbl))$N
    results <- rbind(results, data.frame(method = "adbc", rows = n_label, time_s = adbc_t))
    dbExecute(con, paste("DROP TABLE IF EXISTS", tbl))

    # Snowpark (if available)
    if (has_snowpark) {
      options(RSnowflake.upload_method = "snowpark")
      t0 <- proc.time()
      dbWriteTable(con, tbl, df, overwrite = TRUE)
      sp_t <- (proc.time() - t0)[["elapsed"]]
      cnt <- dbGetQuery(con, paste("SELECT COUNT(*) AS N FROM", tbl))$N
      results <- rbind(results, data.frame(method = "snowpark", rows = n_label, time_s = sp_t))
      dbExecute(con, paste("DROP TABLE IF EXISTS", tbl))
    }
  }
  options(RSnowflake.upload_method = "auto")

  cat("=== Write Benchmark ===\n")
  cat(sprintf("%-10s %5s %8s\n", "Method", "Rows", "Time(s)"))
  cat(strrep("-", 25), "\n")
  for (i in seq_len(nrow(results))) {
    cat(sprintf("%-10s %5s %8.2f\n", results$method[i], results$rows[i], results$time_s[i]))
  }
  if (in_ws) {
    cat("\nIn Workspace, 'auto' routes bulk writes to Snowpark (internal SPCS path).\n")
    cat("ADBC uses the public endpoint. Literal INSERT is fastest for small data.\n")
  }
}

## TEMP: Python Snowpark Write Comparison

**Temporary benchmark cell** -- compares Python `write_pandas` (internal SPCS
path in Workspace) against the R ADBC and literal INSERT results from the
cell above. Also tests Python literal INSERT for a full picture.

Remove this section once the comparison data has been collected.

In [ ]:
import time
import pandas as pd
import numpy as np
from snowflake.snowpark.context import get_active_session

session = get_active_session()
conn = session.connection

# ── write_pandas (PUT + COPY INTO via internal SPCS path) ────────────────────

print("=== Python write_pandas (internal SPCS path) ===")
for n, label in [(5000, "5K"), (50000, "50K")]:
    tbl = f"PY_WRITE_{label}"
    pdf = pd.DataFrame({
        "ID": range(1, n + 1),
        "TXT": [f"row_{i}" for i in range(1, n + 1)],
        "VAL": np.random.rand(n),
    })
    session.sql(f"DROP TABLE IF EXISTS {tbl}").collect()
    t0 = time.time()
    session.write_pandas(pdf, tbl, auto_create_table=True, overwrite=True)
    elapsed = time.time() - t0
    cnt = session.sql(f"SELECT COUNT(*) AS N FROM {tbl}").collect()[0]["N"]
    print(f"  {label:>3} rows: {elapsed:.2f}s  (round-trip: {cnt})")
    session.sql(f"DROP TABLE IF EXISTS {tbl}").collect()

# ── Python literal INSERT (SQL VALUES) ───────────────────────────────────────

print("\n=== Python literal INSERT (SQL VALUES) ===")
cur = conn.cursor()
for n, label in [(5000, "5K"), (50000, "50K")]:
    tbl = f"PY_LIT_{label}"
    cur.execute(f"CREATE OR REPLACE TABLE {tbl} (ID NUMBER, TXT VARCHAR, VAL FLOAT)")
    pdf = pd.DataFrame({
        "ID": range(1, n + 1),
        "TXT": [f"row_{i}" for i in range(1, n + 1)],
        "VAL": np.random.rand(n),
    })
    batch = 16384
    t0 = time.time()
    for start in range(0, n, batch):
        chunk = pdf.iloc[start : start + batch]
        vals = ",".join(
            f"({int(r.ID)},'{r.TXT}',{r.VAL})" for r in chunk.itertuples()
        )
        cur.execute(f"INSERT INTO {tbl} (ID,TXT,VAL) VALUES {vals}")
    elapsed = time.time() - t0
    cnt = cur.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]
    print(f"  {label:>3} rows: {elapsed:.2f}s  (round-trip: {cnt})")
    cur.execute(f"DROP TABLE IF EXISTS {tbl}")
cur.close()

# ── rpy2 in-memory transfer cost (R data.frame → Pandas) ────────────────────

print("\n=== rpy2 R→Pandas in-memory transfer ===")
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter

for n, label in [(5000, "5K"), (50000, "50K")]:
    ro.r(f"tmp_df <- data.frame(ID=seq_len({n}), TXT=paste0('r_',seq_len({n})), VAL=runif({n}))")
    t0 = time.time()
    with localconverter(ro.default_converter + pandas2ri.converter):
        pdf = ro.conversion.get_conversion().rpy2py(ro.r("tmp_df"))
    elapsed = time.time() - t0
    print(f"  {label:>3} rows: {elapsed*1000:.0f}ms")

# ── Summary ──────────────────────────────────────────────────────────────────

print("\n=== Summary (fill in R numbers from cell above) ===")
print("              5K rows     50K rows")
print("  R ADBC:     ___s        ___s       (PUT+COPY, public endpoint)")
print("  R literal:  ___s        ___s       (SQL INSERT)")
print("  Py w_pandas: see above  see above  (PUT+COPY, internal SPCS)")
print("  Py literal:  see above  see above  (SQL VALUES)")
print("  rpy2 xfer:   see above  see above  (in-memory, negligible)")
print()
print("If Py write_pandas is fast, hybrid R→rpy2→write_pandas is viable.")

## 14. Database Object Browsing

`dbListObjects` lets you navigate the Snowflake object hierarchy
(databases, schemas, tables) from R. In RStudio/Positron it also
powers the Connections Pane.

In [ ]:
%%R
# Top level: databases
cat("== Databases (first 5) ==\n")
head(dbListObjects(con), 5)

In [ ]:
%%R
# Drill into the current database -> schemas
db <- Sys.getenv("SNOWFLAKE_DATABASE", "")
if (nzchar(db)) {
  cat("== Schemas in", db, "==\n")
  print(dbListObjects(con, prefix = Id(catalog = db)))
}

In [ ]:
%%R
# Drill into PUBLIC schema -> tables
if (nzchar(db)) {
  cat("== Tables in", db, ".PUBLIC ==\n")
  print(dbListObjects(con, prefix = Id(catalog = db, schema = "PUBLIC")))
}

## 15. Cleanup

In [ ]:
%%R
dbRemoveTable(con, "DEMO_CITIES")
tryCatch(dbExecute(con, "DROP TABLE IF EXISTS ADBC_DEMO"), error = function(e) NULL)
dbDisconnect(con)
cat("Done! Tables removed and connection closed.\n")

## See Also

- **snowflakeR quickstart** (`snowflakeR/inst/notebooks/workspace_quickstart.ipynb`) --
  `sfr_query()`, `sfr_write_table()`, ggplot2 visualization, and `sfr_dbi_connection()`
  for bridging to `RSnowflake` from an `sfr_connection`.
- **Model Registry demo** (`snowflakeR/inst/notebooks/workspace_model_registry.ipynb`) --
  Train, register, deploy, and serve R models via Snowflake Model Registry + SPCS.
- **Forecasting demo** (`snowflakeR/inst/notebooks/workspace_forecasting_demo.ipynb`) --
  ARIMA forecasting with custom `predict_body` template for non-standard model interfaces.